# Install Ultralytics and Roboflow  libs

In [ ]:
!pip install ultralytics roboflow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.0/285.0 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 155.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 7.1 MB/s eta 0:00:00


## Mount Google Drive in Colab Content

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os

BASE_FOLDER = "/content/drive/MyDrive/DentaVision3"
os.makedirs(BASE_FOLDER, exist_ok=True)

print(f"Folder creado: {BASE_FOLDER}")

Folder creado: /content/drive/MyDrive/DentaVision3


## Check Hardware

In [ ]:
import torch

In [ ]:
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

GPU: NVIDIA L4
VRAM (GB): 23.65915136


## Bring Dataset From Roboflow

In [ ]:

from roboflow import Roboflow
rf = Roboflow(api_key="MY-ROBOFLOW-API-KEY")
project = rf.workspace("myworkspace-rde28").project("fdi_numbering-ki0hw")
version = project.version(1)
dataset = version.download("yolov11")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to FDI_numbering-1 in yolov11:: 100%|██████████| 9579/9579 [00:01<00:00, 5694.05it/s]


In [ ]:
# Check dataset is in Content VM disk

In [ ]:
!ls -l {dataset.location}

total 24
-rw-r--r-- 1 root root  526 Aug 10 20:42 data.yaml
-rw-r--r-- 1 root root  138 Aug 10 20:42 README.dataset.txt
-rw-r--r-- 1 root root 1199 Aug 10 20:42 README.roboflow.txt
drwxr-xr-x 4 root root 4096 Aug 10 20:42 test
drwxr-xr-x 4 root root 4096 Aug 10 20:42 train
drwxr-xr-x 4 root root 4096 Aug 10 20:42 valid


In [ ]:
# Import Libraries

In [ ]:
import os
from pathlib import Path
from collections import Counter
import yaml

In [ ]:
# Set yaml file data.yaml  path in local disk

In [ ]:
yaml_path = "/content/FDI_numbering-1/data.yaml"

In [ ]:
with open(yaml_path, "r") as f:
    cfg = yaml.safe_load(f)

## Validate Splits

In [ ]:
names = cfg["names"]
splits = {
    "train": cfg.get("train"),
    "val":   cfg.get("val"),
    "test":  cfg.get("test"),
}

In [ ]:
for split_key, cfg_img_path in splits.items():
    # Extract the directory name for the split (e.g., 'train', 'val', 'test')
    split_dir_name = Path(cfg_img_path).parent.name

    # Construct the full path to the labels directory using dataset.location as base
    label_dir = Path(dataset.location) / split_dir_name / "labels"

    if not label_dir.exists():
        print(f"[{split_key}] Not found : {label_dir}")
        continue

    counter = Counter()
    for lf in label_dir.glob("*.txt"):
        with open(lf) as f:
            for line in f:
                if line.strip():
                    counter[int(line.split()[0])] += 1

    total = sum(counter.values())
    print(f"\n[{split_key}]  {total} instances ")

    for cls_id, name in enumerate(names):
        c = counter.get(cls_id, 0)
        print(f"{name:<10} {c:>6}  {c/total*100 if total else 0:>5.1f}%")



[train]  137532 instances 
Bridge        186    0.1%
Crown         903    0.7%
Implant       234    0.2%
T11          4470    3.3%
T12          4473    3.3%
T13          4503    3.3%
T14          4404    3.2%
T15          4362    3.2%
T16          4344    3.2%
T17          4467    3.2%
T18          2958    2.2%
T21          4449    3.2%
T22          4518    3.3%
T23          4503    3.3%
T24          4404    3.2%
T25          4299    3.1%
T26          4377    3.2%
T27          4452    3.2%
T28          2976    2.2%
T31          4377    3.2%
T32          4401    3.2%
T33          4521    3.3%
T34          4389    3.2%
T35          4431    3.2%
T36          4251    3.1%
T37          4410    3.2%
T38          3387    2.5%
T41          4371    3.2%
T42          4323    3.1%
T43          4503    3.3%
T44          4398    3.2%
T45          4389    3.2%
T46          4284    3.1%
T47          4431    3.2%
T48          3384    2.5%

[val]  5274 instances 
Bridge          2    0.0%
Crown       

In [ ]:
# Author spltis are good , go ahead with convert seg polygon into Bounding Boxes Objects

In [ ]:
import os
from pathlib import Path

## ETL  Transform seg poylogn to Bounding Box

In [ ]:
def convert_seg_to_bbox(label_dir_in, label_dir_out):

    label_dir_in = Path(label_dir_in)
    label_dir_out = Path(label_dir_out)
    label_dir_out.mkdir(parents=True, exist_ok=True)

    for label_file in label_dir_in.glob("*.txt"):

        lines_out = []
        with open(label_file) as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                cls_id = parts[0]
                coords = list(map(float, parts[1:]))

                xs = coords[0::2]
                ys = coords[1::2]

                x_min, x_max = min(xs), max(xs)
                y_min, y_max = min(ys), max(ys)

                x_center = (x_min + x_max) / 2
                y_center = (y_min + y_max) / 2
                width = x_max - x_min
                height = y_max - y_min

                lines_out.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

        with open(label_dir_out / label_file.name, "w") as f:
            f.write("\n".join(lines_out))

    print(f"Convertidos {len(list(label_dir_in.glob('*.txt')))} archivos: {label_dir_in} -> {label_dir_out}")


In [ ]:
# I do not care seg mask for this subject,  i am interested into Object Detection per tooth box

In [ ]:
base = "/content/FDI_numbering-1"
for split in ["train", "valid", "test"]:
    convert_seg_to_bbox(
        f"{base}/{split}/labels",
        f"{base}/{split}/labels_bbox"
    )

Convertidos 4560 archivos: /content/FDI_numbering-1/train/labels -> /content/FDI_numbering-1/train/labels_bbox
Convertidos 175 archivos: /content/FDI_numbering-1/valid/labels -> /content/FDI_numbering-1/valid/labels_bbox
Convertidos 52 archivos: /content/FDI_numbering-1/test/labels -> /content/FDI_numbering-1/test/labels_bbox


In [ ]:
base = "/content/FDI_numbering-1"

In [ ]:
for split in ["train", "valid", "test"]:
    split_dir = Path(base) / split
    labels_orig = split_dir / "labels"
    labels_bbox = split_dir / "labels_bbox"
    labels_seg_backup = split_dir / "labels_seg"

    if labels_orig.exists() and not labels_seg_backup.exists():
        os.rename(labels_orig, labels_seg_backup)
        print(f"{split}: labels -> labels_seg (backup de poligonos)")

    if labels_bbox.exists():
        os.rename(labels_bbox, labels_orig)
        print(f"{split}: labels_bbox -> labels (activo para entrenamiento)")

train: labels -> labels_seg (backup de poligonos)
train: labels_bbox -> labels (activo para entrenamiento)
valid: labels -> labels_seg (backup de poligonos)
valid: labels_bbox -> labels (activo para entrenamiento)
test: labels -> labels_seg (backup de poligonos)
test: labels_bbox -> labels (activo para entrenamiento)


In [ ]:
## Smoke test to see how it is going up on polygons vs boxes / set

In [ ]:
for split in ["train", "valid", "test"]:
    p = Path(base) / split / "labels"
    with open(next(p.glob("*.txt"))) as f:
        print(split, "->", f.readline().strip())

train -> 9 0.327789 0.500977 0.044019 0.194661
valid -> 9 0.277386 0.518555 0.042003 0.180339
test -> 9 0.266681 0.395182 0.046607 0.170573


In [ ]:
## Import YOLO form Ultralytics,
## I am using YOLOv11-small object detection

In [ ]:
from ultralytics import YOLO

In [ ]:
model = YOLO("yolo11s.pt")

In [ ]:
## Set training Hyperparams

In [ ]:
results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=32,
    patience=25,
    optimizer="AdamW",
    lr0=5e-4,
    lrf=0.01,
    weight_decay=5e-4,
    warmup_epochs=3,
    cos_lr=True,
    mosaic=0.0,
    mixup=0.0,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.3,
    degrees=3.0,
    translate=0.05,
    scale=0.3,
    fliplr=0.0,
    flipud=0.0,
    cls=0.7,
    device=0,
    workers=4,
    project = BASE_FOLDER,
    name="fdi_v1_seg",
    exist_ok=True,
    plots=True,
)

Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.7, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/FDI_numbering-1/data.yaml, degrees=3.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.3, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=fdi_v1_seg, nbs=64, nms=Fa

In [ ]:
## Valdiate Model

In [ ]:
metrics = model.val(
    project=BASE_FOLDER,
    name="fdi_v1_seg",
    save_txt=True,
    save_hybrid=True
)

WARNING ⚠️ 'save_hybrid' is deprecated and will be removed in the future.
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLO11s summary (fused): 101 layers, 9,426,345 parameters, 0 gradients, 21.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1682.9±495.7 MB/s, size: 36.4 KB)
val: Scanning /content/FDI_numbering-1/valid/labels.cache... 175 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 175/175 73.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 1.3it/s 8.4s
                   all        175       5274      0.983       0.99       0.99      0.794
                Bridge          2          2      0.982          1      0.995      0.796
                 Crown         20         32      0.926      0.938      0.959      0.756
               Implant          7         10          1       0.97      0.995      0.821
                   T11        173        173      

In [ ]:
## Test Model against the test set

In [ ]:
test_metrics = model.val(
    split='test',
    project=BASE_FOLDER,
    name="fdi_v1_seg",
    save_txt=True,
    save_hybrid=True
)

WARNING ⚠️ 'save_hybrid' is deprecated and will be removed in the future.
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1583.8±447.3 MB/s, size: 35.5 KB)
val: Scanning /content/FDI_numbering-1/test/labels... 52 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 52/52 1.4Kit/s 0.0s
val: New cache created: /content/FDI_numbering-1/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.0s/it 4.0s
                   all         52       1562       0.98      0.967      0.972       0.78
                Bridge          2          2      0.908        0.5      0.495      0.346
                 Crown          8         11      0.773      0.636      0.785       0.58
               Implant          3          4      0.999          1      0.995      0.727
                   T11         50         50      0.996          1      0.995   

In [ ]:
## Gotten Outstandign Results , FDI teeth Model is Good

In [ ]:
TEST_DIR = "/content/images2test"

In [ ]:
## Valdiate model against new images fro mthe web

## Export Model to ONNX -> onxxruntime

In [ ]:
import os

# Export the model. The export method will return the path to the exported file.
exported_model_path = model.export(
    format="onnx",
    imgsz=640,
    opset=18,
    simplify=True
)

# Define the target path and filename
target_onnx_path = os.path.join(BASE_FOLDER, "fdi_best_opset18.onnx")

# Rename and move the exported model to the desired location
os.rename(exported_model_path, target_onnx_path)

print(f"Model exported successfully and moved to: {target_onnx_path}")

Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/content/drive/MyDrive/DentaVision3/fdi_v1_seg/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 39, 8400) (18.3 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 216ms
Prepared 4 packages in 1.50s
Installed 4 packages in 260ms
 + colorama==0.4.6
 + onnx==1.22.0
 + onnxruntime==1.28.0
 + onnxslim==0.1.95

requirements: AutoUpdate success ✅ 2.5s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 18...
ONNX: slimming with onnxslim 0.1.95...
ONNX: export success ✅ 4.8s, saved as 

### Validate Exported ONNX Model

In [ ]:
from ultralytics import YOLO

# Load the exported ONNX model
onnx_model = YOLO(target_onnx_path)

# Validate the ONNX model on the test dataset
# Ensure yaml_path and BASE_FOLDER are defined from previous cells
metrics_onnx = onnx_model.val(
    split='test',
    data=yaml_path,
    project=BASE_FOLDER,
    name="fdi_v1_seg_onnx_validation",
    save_txt=True,
    save_hybrid=True
)

print("ONNX model validation completed.")

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
WARNING ⚠️ 'save_hybrid' is deprecated and will be removed in the future.
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
Loading /content/drive/MyDrive/DentaVision3/fdi_best_opset18.onnx for ONNX Runtime inference...
requirements: Ultralytics requirement ['onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 5 packages in 149ms
Prepared 1 package in 2.38s
Installed 1 package in 67ms
 + onnxruntime-gpu==1.28.0

requirements: AutoUpdate success ✅ 2.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

WARNING ⚠️ CUDA requested but CUDAExecutionProvider not available. Using CPU...
Using ONNX Runtime 1.28.0 with CPUExecutionProvider
Setting batch=1 input of shape (1, 3, 640, 640)
va